# Investigating the impact of the leakage from the Amsterdam Rijnkanaal

Low lying polders adjacent to the Amsterdam Rijnkanaal (ARK) experience more and more
leakage probably from the nearby ARK, which has a water level several meters higher
than that of the polders. The question is what causes this increase of the leagkage occurring
over the last two decades and how can it be solved by use of a layar of a sand-bentione mixture
at the canal bottom.

To investigate this, we'll generate cross section and model them in detail. With this or these
models, different impacts can be simulated and the impact possible measures can be examined for
their effectiveness.

The work is done for Rijkswaterstaat.

@ TO 2026-04-03

In [147]:
import os
import sys
from glob import glob
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pdf2image
import pickle

from mf6lab.Projects.ARK_RWS.src import ARK_fdm

from mf6lab.Projects.ARK_RWS.src.ARK_fdm import (
    geoCodes,
    world_extent,
    CrossSectionDigitizer,
    ImagePicker,
    Dirs,
    plot_result
    )

from tools.fdm.src.mfgrid import Grid

print(sys.executable)

/Users/Theo/Development/python/mf6_tools/mf6lab/.venv/bin/python


# Get the geotop.pdf obtaind from dinoloket.nl

When reading it with pdf2image you get actually two pdfs.
One has the actual cross section and the other the legend and a map.

In [2]:
%matplotlib qt

dirs = Dirs()

geotop_pdf = glob(dirs.dino + '*.pdf')[-1]
geotop1, geotop2 = pdf2image.convert_from_path(geotop_pdf, dpi=300)
geotop1 = np.asarray(geotop1.convert("RGB"))  
geotop2 = np.asarray(geotop2.convert("RGB"))   

# Use the image picker with the legend image to get the legend colors

Instantiate the ImagePicker with the image that holds the map with the legend color boxes.
Then zoom in and click the legend's color boxes in the normal order.
This yields the colors to compare those sampled in the actual cross section with.

Press ENTER to finish with the legend.

In [3]:
# ---Instantiate the picker with the image to pick from (image with the legend)
picker = ImagePicker(geotop2)

# --- Zoom before clicking (zoom into legend)
# --- Step 1: Pick legend colors
legend_colors = picker.get_colors(n=-1)
print("Legend_colors: ", legend_colors)

# --- The last legend_color is always [255., 255., 255.] to indicate empty cells later on
# --- Therefore we add a legend index 'none' for this code.
geoCodes.append('none')

Legend_colors:  [[118. 147.  60.]
 [102. 205. 171.]
 [170. 255. 245.]
 [208. 130.  40.]
 [152.  47.  10.]
 [255. 255.  80.]
 [255. 235.   0.]
 [255. 127.  80.]
 [156. 156. 156.]
 [255. 255. 255.]]


# With the legend_colors now obtained get the pixel extent of the cross section

Instansiate the ImagePicker one again, but now with the image of the actual cross section.
Zoom in (or presse the image window to the top of the screen to make the window full screen). Then pick the LL nd UR of the cross section of which you know the world coordinates.

Press ENTER when finished. The pixel_exent is thus obtained.

In [4]:
# --- Initiate a new picker, now with the cross section   
# --- To get the pixel bounding box
picker = ImagePicker(geotop1)

# --- Again, zoom in
# --- Then pick 2 opposite corners, return to finish
extent = picker.get_bbox(n=-1)
print(extent)

[(366, 1703), (2967, 402)]
[(366, 1703), (2967, 402)]
(366, 2967, 402, 1703)
(366, 2967, 402, 1703)


# Sample the actual cross section automatically

1. Instantiate the CrossSectionDigitizer with the image of the actual cross section.
1. Make sure it knows its extent (pixel extent)
1. Make sure it knows its world_extent (LL and UR corners)
1. Make sure it knows the legend colors.
1. Make sure it know its grid (Voxel size)

Set the grid by specifying the voxel width dx and the voxel height dz.
These size dx, dz should be obtained from the image. Note that the
real voxel size of geotop is dx=dy=100 m and dz=0.5 m. But when the
the X-sec is neither parallel to the x or y axis, the dx may be different
along the length of the X-section.

The subdivision of the grid is based on the world_extent and the voxel size.

In [5]:
# --- Fill an array with soil indices where each index is the number of the
# --- legend color boxes in that order (the order clicked before)

# --- Instantiate the CrossSectionDigitizer using the cross section.
digitizer = CrossSectionDigitizer(image=geotop1)

# --- Set pixed extent (see extent obtained above)
digitizer.set_section_bbox(*extent)

# --- Set world extent (given above in world coordinates)
digitizer.set_world_bbox(*world_extent)

# --- Internally set the legend_colors (obtained from clicking the legend)
digitizer.legend_colors = legend_colors

# --- Set voxel size.
digitizer.set_grid(dx=100, dz=0.5)

# Sample the cross section

The digitizer's build_array is used to sample the cross section.

A small area around the centre of each voxel is sampled. Dark colors are removed
to avoid text and lines and the median color (RGB) is sampled. The distance in RGB
space to the legend color is computed and the nearest legend color is found, which
yield the number of the legend box corresponding to the sampled color.

The highest number corresponds to color [255., 255., 255.] meaning no color.

The resulting voxel X-sec array with legend indices can always be converted to
any property that can be linked to the legend. The best way to do this is by
using a pandas DataFrame with the same index as the legend colorboxes.

An mfgrid.Grid object can be used as an alternative grid, but this is not
recommended because it can create artefacts: When the resolution is too fine
these artefacts are caused by lines and text in the image.

Resampling on other grids should be done separately.

In [6]:
# === Fill in an array of a cross section using  dx and dz and world_extent
arr = digitizer.build_array()

In [7]:
# --- Show the cross section using imshow, which fills the voxels
plot_result(arr, world_extent)

# --- Add title and save
fig = plt.gcf()
fig.suptitle(f"""{os.path.basename(geotop_pdf)}
                with colors converted to soil-indices
                """)
fig.savefig(os.path.join(dirs.images, f"{os.path.basename(geotop_pdf)}"))

plt.show()

## Get the legend colors of each geotop X-section

The legend is the second page of each geotop pdf file. Each
image is loaded in turn and the colors of the legend are clicked
and stored in a dictionary whose keyse are the basename of the
mentioned files.

These colors should be matched with the labels of the legends.
But these labels have to be provided by the user and are
given below for the current geotop pdfs


In [ ]:
# --- Coordinate data for the various cross sections

# --- Dictionary to store the picked points (pixels)
geotopLegends = {}

# --- Run over all the geotop files stored.
for geotop_pdf in glob(dirs.dino + '*.pdf'):
    
    # --- Use  basename as key.
    basename = os.path.basename(geotop_pdf)
    
    # --- Get the two pages of each geotop pdf as RGB image 
    geotop_xsec, geotop_leg = pdf2image.convert_from_path(geotop_pdf, dpi=300)
        
    geotop_xsec = np.asarray(geotop_xsec.convert("RGB"))  # map
    geotop_leg  = np.asarray(geotop_leg.convert("RGB"))  # legend (not used here) 
    
    # --- Instantiate the point picker with the map image
    picker = ImagePicker(geotop_leg)
    
    # --- Click the 5 points
    legend_colors = picker.get_colors(n=-1)

    # --- Store them
    geotopLegends[basename] = legend_colors
    
    

In [148]:
labels = ["NUAAOP	NUECgb	NUEC1	NUNIHO	NUNIBA	NUNBXWI-SI-KO	NUBX	NUKR-BXDE	NUDR	Nugs	NUUR2	NUST",
"a	v	k	kz	zf	zm	zg	g	she",
"a	v	k	kz	zf	zm	zg	g	she",		
"NUECga	NUECgb	NUEC1	NUNIHO	NUNIBA	NUNBXWI-SI-KO	NUBX	NUKR-BXDE	NUDR	NUgs	NUUR2	NUST",
"NUAAOP	NUECga	NUECgb	NUEC1	NUNIHO	NUNIBA	NUNBXWI-SI-KO	NUBX	NUDR	NUgs	NUST",
"NUAAOP	NUECga	NUECgb	NUEC1	NUNIHO	NUNIBA	NUNBXWI-SI-KO	NUBX	NUKR-BXDE	NUDR	NUgs	NUUR2	NUST",
"NUAAOP	NUECgb	NUEC1	NUNIHO	NUNIBA	NUNBXWI-SI-KO	NUBX	NUKR-BXDE	NUDR	NUgs	NUUR2	NUST",
"a	v	k	kz	zf	zm	zg	g	she",
"a	v	k	kz	zf	zm	zg	g	she",
"a	v	k	kz	zf	zm	zg	g	she"
]

geolegs = {}

for (fname, colors), label in zip(geotopLegends.items(), labels):
    geolegs[fname] = {'labels': label.split('\t') + ['none'], 'colors':colors}
    

pkl_fname = os.path.join(dirs.data, 'geotop_legends.pkl')

with open(pkl_fname, 'wb') as f:
    pickle.dump(geolegs, f)
print(f"geolegs pickled to file {os.path.basename(pkl_fname)}")
print(f"in directory: {dirs.data}")
    
geolegs


geolegs pickled to file geotop_legends.pkl
in directory: /Users/Theo/Development/python/mf6_tools/mf6lab/Projects/ARK_RWS/data/


{'BRO GeoTOP Verticale doorsnede geologische eenheid 130173,479431.pdf': {'labels': ['NUAAOP',
   'NUECgb',
   'NUEC1',
   'NUNIHO',
   'NUNIBA',
   'NUNBXWI-SI-KO',
   'NUBX',
   'NUKR-BXDE',
   'NUDR',
   'Nugs',
   'NUUR2',
   'NUST',
   'none'],
  'colors': array([[200., 200., 200.],
         [102., 205., 171.],
         [170., 255., 245.],
         [208., 130.,  40.],
         [152.,  47.,  10.],
         [255., 255.,  80.],
         [255., 235.,   0.],
         [176.,  48.,  96.],
         [255., 127.,  80.],
         [156., 156., 156.],
         [189., 183., 107.],
         [205.,  92.,  92.],
         [255., 255., 255.]])},
 'BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 126311,474187.pdf': {'labels': ['a',
   'v',
   'k',
   'kz',
   'zf',
   'zm',
   'zg',
   'g',
   'she',
   'none'],
  'colors': array([[200., 200., 200.],
         [157.,  78.,  64.],
         [  0., 146.,   0.],
         [194., 207.,  92.],
         [255., 255.,   0.],
         [243., 225

# Get the point data necessary to georeference the different cross sections

Now that we practiced sampling a Geotop X-ref image, we can do it for a larger set of geotop x-sections.

But before doing that, we will get the extent of each cross section in both pixel and real-world coordinates with z NAP and x measured along the cross section.

To get the extent of each x-section:

Each section is loaded in turn. For each section click on three points of the left z-axis
followed by three points on the bottom x-axis. The points to be clicked are the zero point
of the axis, the last tick of the axis and the bottom/end of the colored section, which
is the actual bottom and the actual end of the colored section. Together with the values
on the axes, to be read separately, the georeferencing can be uniquely done in terms
of the vertical NAP  coordinate and the horizontal coordinate along the section.

In [150]:
# --- Coordinate data for the various cross sections

# --- Dictionary to store the picked points (pixels)
geotopPtsData = {}

# --- Run over all the geotop files stored.
for geotop_pdf in glob(dirs.dino + '*.pdf'):
    
    # --- Use  basename as key.
    basename = os.path.basename(geotop_pdf)
    
    # --- Get the two pages of each geotop pdf as RGB image 
    geotop_xsec, geotop_leg = pdf2image.convert_from_path(geotop_pdf, dpi=300)
        
    geotop_xsec = np.asarray(geotop_xsec.convert("RGB"))  # map
    geotop_leg  = np.asarray(geotop_leg.convert("RGB"))  # legend (not used here) 
    
    # --- Instantiate the point picker with the map image
    picker = ImagePicker(geotop_xsec)
    
    # --- Click the 5 points
    points = picker.pick_points(n=-1)

    # --- Store them
    geotopPtsData[basename] = points
    

##  Convert the points for each X-sec into a dict with keys denoting their meaning

For each geotop X-section, 6 coordinates have now been collected into the
geotopPtsData dictionary. The keys are the basenames of the geotop pdf files, and the values is a list of 5 clicked pixel coordinate pairs: 3 along the z axis, followed by
2 along the x-axes. (Because the first point on the x-axis is the same as the
third point of the z-axis, only 5 points have to be clicked).

The 3 clicked values along the z-axis are z=0, z=lowest tick and z=bottom of the X-section.
The 2 clicked values along the x-axis are x=largest tick and x=right end of the X-section

Next is to convert these 5 coordinate pairs into 10 numbers:
xp1, zp1, xp2, zp2, xp3, zp3, xp4, zp4, xp5, zp5

Then put them in a dictionary, with keys denoting their meaning.

This allows easy conversion into the columns of a pd.DataFrame with
basenames as index.

In [151]:
# --- The dictionary (copy to not destroy the captured points and use a shorter name)
db = geotopPtsData.copy()

# --- Convert the collected coordinate pairs into the z and x coordinates
for k in db.keys():
    
    # --- first the 6 coordinate pairs of each geotop X-section --> array
    pts = np.array(db[k])
    
    # --- Select the 3 z-values and the 3 x-values
    db[k] = {'xp1': pts[0, 0], 'zp1': pts[0, 1],
             'xp2': pts[1, 0], 'zp2': pts[1, 1],
             'xp3': pts[2, 0], 'zp3': pts[2, 1],
             'xp4': pts[3, 0], 'zp4': pts[3, 1],
             'xp5': pts[4, 0], 'zp5': pts[4, 1]}

# --- Generate a DataFrame to hold the extents of all the geotop_pdf files
extents = pd.DataFrame(index=db.keys(),
                          columns=db[k].keys()
                          )

# --- Fill it with the captured points
for k in db.keys():   
    extents.loc[k] = db[k]

# --- Make sure their type is not object, but int.
for col in extents.columns:
    extents[col] = extents[col].astype(int)
    
# --- Show the DataFrame
extents

,xp1,zp1,xp2,zp2,xp3,zp3,xp4,zp4,xp5,zp5
"BRO GeoTOP Verticale doorsnede geologische eenheid 130173,479431.pdf",364,402,366,1575,366,1707,2887,1705,2962,1707
"BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 126311,474187.pdf",368,400,366,1568,366,1705,2882,1705,2959,1705
"BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 130049,479466.pdf",366,385,366,1565,366,1703,2822,1703,2967,1705
"BRO GeoTOP Verticale doorsnede geologische eenheid 129926,479462.pdf",330,373,330,1647,332,1705,2858,1705,2962,1705
"BRO GeoTOP Verticale doorsnede geologische eenheid 127484,477893.pdf",361,397,366,1570,364,1703,2801,1705,2962,1705
"BRO GeoTOP Verticale doorsnede geologische eenheid 130049,479466.pdf",366,385,364,1568,364,1703,2825,1705,2964,1703
"BRO GeoTOP Verticale doorsnede geologische eenheid 126311,474187.pdf",368,402,364,1573,366,1703,2880,1703,2962,1705
"BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 129926,479462.pdf",332,376,332,1650,330,1705,2856,1705,2962,1705
"BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 127484,477893.pdf",366,397,364,1573,364,1705,2801,1705,2959,1705
"BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 130173,479431.pdf",366,397,366,1573,364,1703,2887,1705,2962,1705


## Add real world values of Pz1 (z1) and Pz2 (z2) as well as Px1 (x1) and Px2 (x2)

The world values of pixels Pz1, Pz2, Px1 and Px2 are provided by the user and are
given below. The values have been obtained by clicking the points using the
ImagePicker in a loop running over each geotop.pdf file in the dirs.dino directory.

When done, these two DataFrames are merged and the world_extent values z3 and x3 are
computed and then added to the DataFrame.

In [152]:
# --- Coordinates z1, z2, x1, x2 of the geotop cross sections
wrld_axes = pd.DataFrame(data=np.array([
                      [0.0, -45.0, -99.0, 0.0, 8400.0, 9999.0],
                      [0.0, -45.0, -99.0, 0.0, 5600.0, 9999.0],
                      [0.0, -45.0, -99.0, 0.0, 8250.0, 9999.0],
                      [0.0, -48.0, -99.0, 0.0, 7000.0, 9999.0],
                      [0.0, -45.0, -99.0, 0.0, 4800.0, 9999.0],
                      [0.0, -45.0, -99.0, 0.0, 8250.0, 9999.0],
                      [0.0, -45.0, -99.0, 0.0, 5600.0, 9999.0],
                      [0.0, -48.0, -99.0, 0.0, 7000.0, 9999.0],
                      [0.0, -45.0, -99.0, 0.0, 4800.0, 9999.0],
                      [0.0, -45.0, -99.0, 0.0, 8400.0, 9999.0]
                      ]),
             index=db.keys(),
             columns = ['z1', 'z2', 'z3', 'x1', 'x2', 'x3']
)

# --- Merge the two DataFrames
for col in wrld_axes.columns:
    extents[col] = wrld_axes[col]

# --- Show the extents DataFrame
extents

,xp1,zp1,xp2,zp2,xp3,zp3,xp4,zp4,xp5,zp5,z1,z2,z3,x1,x2,x3
"BRO GeoTOP Verticale doorsnede geologische eenheid 130173,479431.pdf",364,402,366,1575,366,1707,2887,1705,2962,1707,0.0,-45.0,-99.0,0.0,8400.0,9999.0
"BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 126311,474187.pdf",368,400,366,1568,366,1705,2882,1705,2959,1705,0.0,-45.0,-99.0,0.0,5600.0,9999.0
"BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 130049,479466.pdf",366,385,366,1565,366,1703,2822,1703,2967,1705,0.0,-45.0,-99.0,0.0,8250.0,9999.0
"BRO GeoTOP Verticale doorsnede geologische eenheid 129926,479462.pdf",330,373,330,1647,332,1705,2858,1705,2962,1705,0.0,-48.0,-99.0,0.0,7000.0,9999.0
"BRO GeoTOP Verticale doorsnede geologische eenheid 127484,477893.pdf",361,397,366,1570,364,1703,2801,1705,2962,1705,0.0,-45.0,-99.0,0.0,4800.0,9999.0
"BRO GeoTOP Verticale doorsnede geologische eenheid 130049,479466.pdf",366,385,364,1568,364,1703,2825,1705,2964,1703,0.0,-45.0,-99.0,0.0,8250.0,9999.0
"BRO GeoTOP Verticale doorsnede geologische eenheid 126311,474187.pdf",368,402,364,1573,366,1703,2880,1703,2962,1705,0.0,-45.0,-99.0,0.0,5600.0,9999.0
"BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 129926,479462.pdf",332,376,332,1650,330,1705,2856,1705,2962,1705,0.0,-48.0,-99.0,0.0,7000.0,9999.0
"BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 127484,477893.pdf",366,397,364,1573,364,1705,2801,1705,2959,1705,0.0,-45.0,-99.0,0.0,4800.0,9999.0
"BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 130173,479431.pdf",366,397,366,1573,364,1703,2887,1705,2962,1705,0.0,-45.0,-99.0,0.0,8400.0,9999.0


## Add the z-extent and x-extent columns to the database

The z- adn x-extent are computed

$$ z_3 = z_1 + \frac{zp_3 - {zp_1}}{zp_2 - zp_1} (z_2 - z_1) $$

$$ x_3 = x_1 + \frac{xp_3 - {xp_1}}{xp_2 - xp_1} (x_2 - x_1) $$


In [154]:
# --- Compute the bottom coordinate of the z-axis
extents['z3'] = (extents['z1'] + 
              (extents['zp3'] - extents['zp1']) / (extents['zp2'] - extents['zp1']) *
              (extents['z2'] - extents['z1'])
)

# --- Compute the largest coordinate of the x-axis
extents['x3'] = (extents['x1'] + 
              (extents['xp3'] - extents['xp1']) / (extents['xp2'] - extents['xp1']) *
              (extents['x2'] - extents['x1'])
)

# --- Just round for convenience
extents['x3'] = np.round(extents['x3'])

# --- Show the extents DataFrame, which is now complete
extents

,xp1,zp1,xp2,zp2,xp3,zp3,xp4,zp4,xp5,zp5,z1,z2,z3,x1,x2,x3
"BRO GeoTOP Verticale doorsnede geologische eenheid 130173,479431.pdf",364,402,366,1575,366,1707,2887,1705,2962,1707,0.0,-45.0,-50.063939,0.0,8400.0,8400.0
"BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 126311,474187.pdf",368,400,366,1568,366,1705,2882,1705,2959,1705,0.0,-45.0,-50.278253,0.0,5600.0,5600.0
"BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 130049,479466.pdf",366,385,366,1565,366,1703,2822,1703,2967,1705,0.0,-45.0,-50.262712,0.0,8250.0,NaN
"BRO GeoTOP Verticale doorsnede geologische eenheid 129926,479462.pdf",330,373,330,1647,332,1705,2858,1705,2962,1705,0.0,-48.0,-50.185243,0.0,7000.0,inf
"BRO GeoTOP Verticale doorsnede geologische eenheid 127484,477893.pdf",361,397,366,1570,364,1703,2801,1705,2962,1705,0.0,-45.0,-50.102302,0.0,4800.0,2880.0
"BRO GeoTOP Verticale doorsnede geologische eenheid 130049,479466.pdf",366,385,364,1568,364,1703,2825,1705,2964,1703,0.0,-45.0,-50.135249,0.0,8250.0,8250.0
"BRO GeoTOP Verticale doorsnede geologische eenheid 126311,474187.pdf",368,402,364,1573,366,1703,2880,1703,2962,1705,0.0,-45.0,-49.995730,0.0,5600.0,2800.0
"BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 129926,479462.pdf",332,376,332,1650,330,1705,2856,1705,2962,1705,0.0,-48.0,-50.072214,0.0,7000.0,-inf
"BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 127484,477893.pdf",366,397,364,1573,364,1705,2801,1705,2959,1705,0.0,-45.0,-50.051020,0.0,4800.0,4800.0
"BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 130173,479431.pdf",366,397,366,1573,364,1703,2887,1705,2962,1705,0.0,-45.0,-49.974490,0.0,8400.0,-inf


## Save the extents DataFrame to a pickle file for later retrieval

In [155]:
pkl_file = os.path.join(dirs.data, 'geotop_extents.pkl')

extents.to_pickle(pkl_file)

print(f"Geotop_pdf's extent saved to {pkl_file}")
print(f"That is to file {os.path.basename(pkl_file)}")
print(f"In directory {dirs.dino}")

Geotop_pdf's extent saved to /Users/Theo/Development/python/mf6_tools/mf6lab/Projects/ARK_RWS/data/geotop_extents.pkl
That is to file geotop_extents.pkl
In directory /Users/Theo/Development/python/mf6_tools/mf6lab/Projects/ARK_RWS/data/dinoloket/


## Sampling the geotop X-ssection voxel arrays

We are now ready to sample the cross section's voxels and match them with their
legend. For each geotop X-section file, we first click the extents of the
X-section and then match them with its world_extents, which is know from the
extents file.

The easiest way is to automatically fill the arrays, which is possible when we
store for each cross section the colors of the legend and the X-section
pixel-extent and world-extent.


In [161]:
geotop_xsec = {}

# --- Run over all the geotop files stored.
for geotop_pdf in glob(dirs.dino + '*.pdf'):
    fname = os.path.basename(geotop_pdf)
    rec = extents.loc[fname]
    pixel_extent = (rec['xp1'], rec['xp5'], rec['zp3'], rec['zp1'])
    world_extent = (rec['x1'],  rec['x3'],  rec['z3'],  rec['z1'])
    
    xsec_image = pdf2image.convert_from_path(geotop_pdf)[0]
    
    digitizer = CrossSectionDigitizer(xsec_image)
    
    digitizer.legend_colors = legend_colors
    digitizer.set_section_bbox(*pixel_extent)
    digitizer.set_world_bbox(*world_extent)
    digitizer.set_grid(dx=100., dz=0.5)
    arr = digitizer.build_array()
    
    geotop_xsec[k] = {'extent': pixel_extent,
                      'world_extent': world_extent,
                      'colors': geolegs[fname]['colors'],
                      'labels': geolegs[fname]['labels'],
                      'shape': arr.shape,
                      'idx': arr,
                      }
    
    plot_result(arr, world_extent=world_extent)


AttributeError: 'PpmImageFile' object has no attribute 'shape'

In [159]:
rec

xp1     364.000000
zp1     402.000000
xp2     366.000000
zp2    1575.000000
xp3     366.000000
zp3    1707.000000
xp4    2887.000000
zp4    1705.000000
xp5    2962.000000
zp5    1707.000000
z1        0.000000
z2      -45.000000
z3      -50.063939
x1        0.000000
x2     8400.000000
x3     8400.000000
Name: BRO GeoTOP Verticale doorsnede geologische eenheid 130173,479431.pdf, dtype: float64